# Historical Milestone 3 Sampling and Annotation

This notebook preserves already completed sampling preparation and the
historical annotation procedure. It is not required for the current workflow.
All former code cells are stored as non-executable examples to prevent
accidental file generation or review changes. No cells were rerun.
Manual review is still pending; assignments and automated suggestions do not
represent completed human reviews.

Use `03_model_training_and_comparison.ipynb` for the current workflow.
It loads `data/labeled/suitability_review_master_v1.csv` and
`data/labeled/suitability_development_labels_v1.csv`, with sampling metadata
read from the master. Older CSV references below point to
`data/labeled/archive/` and are historical only.


### Historical step 1 — already completed / reference only

Not required for the current workflow. Do not execute this saved example.

```python
from pathlib import Path
import pandas as pd

# Locate the project root.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data" / "raw").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / "data" / "raw"

if not DATA_DIR.is_dir():
    raise FileNotFoundError(f"Raw data folder not found: {DATA_DIR}")
```


### Historical step 2 — already completed / reference only

Not required for the current workflow. Do not execute this saved example.

```python
# Load only the columns needed for Milestone 3.
# Exclude the large precomputed embedding columns.

cv = pd.read_csv(
    DATA_DIR / "cv.csv",
    usecols=[
        "candidate_id",
        "resume_text",
        "clean_skills",
        "inferred_category",
    ],
)

jobs = pd.read_csv(
    DATA_DIR / "job.csv",
    usecols=[
        "job_id",
        "job_title",
        "job_text",
        "clean_skills",
        "category",
    ],
)

matches = pd.read_csv(
    DATA_DIR / "matches.csv",
    usecols=[
        "candidate_id",
        "job_id",
        "label_strength",
    ],
)

print("DATASET SIZES")
print(f"Resumes: {len(cv):,}")
print(f"Jobs: {len(jobs):,}")
print(f"Resume-job pairs: {len(matches):,}")

print("\nMISSING VALUES IN LOADED COLUMNS")
for name, frame in [
    ("Resumes", cv),
    ("Jobs", jobs),
    ("Matches", matches),
]:
    print(f"{name}: {int(frame.isna().sum().sum()):,}")

print("\nDUPLICATE IDENTIFIERS AND PAIRS")
print("Duplicate candidate IDs:", int(cv["candidate_id"].duplicated().sum()))
print("Duplicate job IDs:", int(jobs["job_id"].duplicated().sum()))
print(
    "Duplicate resume-job pairs:",
    int(matches.duplicated(["candidate_id", "job_id"]).sum()),
)

print("\nUNKNOWN REFERENCES")
print(
    "Pairs with unknown candidate IDs:",
    int((~matches["candidate_id"].isin(cv["candidate_id"])).sum()),
)
print(
    "Pairs with unknown job IDs:",
    int((~matches["job_id"].isin(jobs["job_id"])).sum()),
)

print("\nMATCH LABEL DISTRIBUTION")
label_counts = matches["label_strength"].value_counts(dropna=False)

label_summary = pd.DataFrame({
    "Count": label_counts,
    "Percentage": (label_counts / len(matches) * 100).round(2),
})

print(label_summary.to_string())

print("\nCANDIDATE CATEGORY DISTRIBUTION")
print(cv["inferred_category"].value_counts(dropna=False).to_string())

print("\nStep 2 completed. No files were changed.")
```

Historical saved output (not rerun):

```text
DATASET SIZES
Resumes: 9,544
Jobs: 1,167
Resume-job pairs: 122,560

MISSING VALUES IN LOADED COLUMNS
Resumes: 0
Jobs: 0
Matches: 0

DUPLICATE IDENTIFIERS AND PAIRS
Duplicate candidate IDs: 0
Duplicate job IDs: 0
Duplicate resume-job pairs: 0

UNKNOWN REFERENCES
Pairs with unknown candidate IDs: 0
Pairs with unknown job IDs: 0

MATCH LABEL DISTRIBUTION
                 Count  Percentage
label_strength                    
medium_positive  70636       57.63
weak_positive    51150       41.73
strong_positive    774        0.63

CANDIDATE CATEGORY DISTRIBUTION
inferred_category
INFORMATION-TECHNOLOGY    8140
FINANCE                   1010
SALES                      215
BUSINESS-DEVELOPMENT        83
HR                          80
UNKNOWN                     16

Step 2 completed. No files were changed.
```


### Historical step 3 — already completed / reference only

Not required for the current workflow. Do not execute this saved example.

```python
import numpy as np

# Create folders for the new Milestone 3 work.
LABEL_DIR = PROJECT_ROOT / "data" / "labeled"
REPORT_DIR = PROJECT_ROOT / "reports" / "milestone3"
MODEL_DIR = PROJECT_ROOT / "models" / "milestone3"

for folder in [LABEL_DIR, REPORT_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

review_path = LABEL_DIR / "archive" / "suitability_review_v1.csv"
sampling_path = LABEL_DIR / "archive" / "sampling_details_v1.csv"

# Protect any review work you have already saved.
if review_path.exists() or sampling_path.exists():
    raise FileExistsError(
        "The review files already exist. Keep them rather than overwriting them."
    )

rng = np.random.default_rng(42)

job_domains = jobs.set_index("job_id")["category"]
observed_pairs = set(
    zip(matches["candidate_id"], matches["job_id"])
)

selected = []
used_pairs = set()

for domain in sorted(jobs["category"].dropna().unique()):

    domain_matches = matches[
        matches["job_id"].map(job_domains).eq(domain)
    ]

    # Select two examples from each existing positive-strength category.
    for strength in [
        "strong_positive",
        "medium_positive",
        "weak_positive",
    ]:
        pool = domain_matches[
            domain_matches["label_strength"].eq(strength)
        ]

        sample = pool.sample(
            n=min(2, len(pool)),
            random_state=42,
        )

        for row in sample.itertuples(index=False):
            pair = (row.candidate_id, int(row.job_id))

            if pair not in used_pairs:
                selected.append({
                    "candidate_id": pair[0],
                    "job_id": pair[1],
                    "sampling_group": strength,
                    "sampling_domain": domain,
                })
                used_pairs.add(pair)

    # Select four unrecorded pairs within the same supplied domain.
    # Their suitability remains unknown until review.
    candidate_ids = cv.loc[
        cv["inferred_category"].eq(domain),
        "candidate_id",
    ].to_numpy()

    job_ids = jobs.loc[
        jobs["category"].eq(domain),
        "job_id",
    ].to_numpy()

    added = 0

    if len(candidate_ids) and len(job_ids):
        for _ in range(10_000):
            if added == 4:
                break

            pair = (
                str(rng.choice(candidate_ids)),
                int(rng.choice(job_ids)),
            )

            if pair in observed_pairs or pair in used_pairs:
                continue

            selected.append({
                "candidate_id": pair[0],
                "job_id": pair[1],
                "sampling_group": "unrecorded_same_domain",
                "sampling_domain": domain,
            })
            used_pairs.add(pair)
            added += 1

sampling = pd.DataFrame(selected)

sampling["pair_id"] = (
    sampling["candidate_id"].astype(str)
    + ":"
    + sampling["job_id"].astype(str)
)

# Attach the actual documents for reviewers to read.
review = (
    sampling[["pair_id", "candidate_id", "job_id"]]
    .merge(
        cv[["candidate_id", "resume_text"]],
        on="candidate_id",
        how="left",
        validate="many_to_one",
    )
    .merge(
        jobs[["job_id", "job_title", "job_text"]],
        on="job_id",
        how="left",
        validate="many_to_one",
    )
)

assert not review[["resume_text", "job_text"]].isna().any().any()
assert not review["pair_id"].duplicated().any()

# Leave all suitability labels and reviewer identities blank.
for column in [
    "reviewer_1",
    "label_1",
    "reviewer_2",
    "label_2",
    "final_label",
    "rationale",
    "adjudication_note",
]:
    review[column] = ""

review["review_status"] = "pending"

# Keep sampling information separate to reduce reviewer anchoring.
review.to_csv(review_path, index=False, encoding="utf-8-sig")
sampling.to_csv(sampling_path, index=False, encoding="utf-8-sig")

print(f"Prepared {len(review)} pairs for review.")
print("No suitability labels have been assigned.")

print("\nReview file:")
print(review_path)

print("\nSampling details:")
print(sampling_path)

print("\nSampling counts:")
print(
    sampling.groupby(
        ["sampling_domain", "sampling_group"]
    ).size().to_string()
)
```


## Suitability Annotation Rubric

### Purpose

Assign a suitability label to each resume–job pair using evidence from
the resume and job description.

### Label Definitions

**1 — Suitable Match**

The resume demonstrates the core skills and relevant experience needed
for the role, with no clearly unmet essential requirement. Relevant
transferable skills may count when the reviewer explains the equivalence.
Missing optional or preferred skills alone do not make a pair unsuitable.

**0 — Unsuitable Match**

The resume shows a substantial mismatch with the role's core duties or
clearly does not meet an essential requirement. The reviewer must identify
the specific requirement and supporting evidence.

**Uncertain**

Leave the label blank when information is insufficient, contradictory,
or ambiguous. Record the reason and keep the pair pending. An omitted
skill phrase alone does not prove that the candidate lacks the skill.

### Review Rules

- Read the full resume and job description.
- Assess job-relevant skills, responsibilities, and experience.
- Distinguish essential requirements from optional preferences.
- Do not use personal or protected characteristics.
- Do not use source match strengths, similarity scores, or skill-coverage
  percentages as ground-truth suitability labels.
- Do not assume that an unrecorded pair is unsuitable.
- Record a short evidence-based rationale for each decision.

### Independent Review

A primary reviewer labels every pair. A second reviewer independently
labels a subset selected before annotation, without seeing the first
reviewer's decisions.

Disagreements are discussed using the rubric. Record the final decision
and its justification in the final_label, rationale, and
adjudication_note columns.

Keep unresolved pairs pending. Only completed, resolved labels may enter
the supervised training dataset.

### Interpretation

Labels describe suitability under this rubric. They do not represent
actual hiring decisions or probabilities of being hired.

### Historical step 5 — already completed / reference only

Not required for the current workflow. Do not execute this saved example.

```python
# Reload the latest saved review records.
review = pd.read_csv(review_path, keep_default_na=False)

pair_id = "cand_04133:3891073377"
matching_rows = review.index[review["pair_id"].eq(pair_id)]

if len(matching_rows) != 1:
    raise ValueError("Expected exactly one matching pair.")

row_index = matching_rows[0]

if str(review.at[row_index, "label_1"]).strip():
    raise ValueError("This pair already has a first-review label.")

reviewer = input("Your reviewer name: ").strip()
decision = input("Your decision: 0, 1, or U for uncertain: ").strip().upper()
rationale = input("Your evidence-based rationale: ").strip()

if not reviewer or not rationale:
    raise ValueError("Reviewer name and rationale are required.")

if decision not in {"0", "1", "U"}:
    raise ValueError("Enter 0, 1, or U.")

review.at[row_index, "reviewer_1"] = reviewer
review.at[row_index, "label_1"] = "" if decision == "U" else decision
review.at[row_index, "rationale"] = rationale

# Final label remains blank until the review process is complete.
review.at[row_index, "review_status"] = "pending"

review.to_csv(review_path, index=False, encoding="utf-8-sig")

print("First-review decision saved for:", pair_id)
```


### Display the Next Pair for Review

This cell opens the latest review file and finds the first resume–job
pair that has neither a reviewer name nor a first-review label.

It displays the pair ID, job title, full resume, and job description
so we can read both documents and assess the match using our rubric.

It skips pairs we have already reviewed, including those marked
uncertain with a reviewer name. If no untouched pairs remain, it
prints a message saying every pair has received an initial review.

This cell only displays information. It does not assign labels or
change the saved file.

### Historical step 7 — already completed / reference only

Not required for the current workflow. Do not execute this saved example.

```python
review = pd.read_csv(review_path, keep_default_na=False)

pending = review[
    review["label_1"].astype(str).str.strip().eq("")
    & review["reviewer_1"].astype(str).str.strip().eq("")
]

if pending.empty:
    print("Every pair has received an initial review.")
else:
    pair = pending.iloc[0]
    pair_id = pair["pair_id"]

    print("PAIR ID:", pair_id)
    print("JOB TITLE:", pair["job_title"])

    print("\nRESUME\n")
    print(pair["resume_text"])

    print("\nJOB DESCRIPTION\n")
    print(pair["job_text"])
```


### Historical step 8 — already completed / reference only

Not required for the current workflow. Do not execute this saved example.

```python
review_path = (
    PROJECT_ROOT / "data" / "labeled" / "archive" / "suitability_review_v1.csv"
)

review = pd.read_csv(review_path, keep_default_na=False)

pending = review[
    review["label_1"].astype(str).str.strip().eq("")
    & review["reviewer_1"].astype(str).str.strip().eq("")
]

if pending.empty:
    print("Every pair has received an initial review.")
else:
    pair = pending.iloc[0]

    print("PAIR ID:", pair["pair_id"])
    print("JOB TITLE:", pair["job_title"])

    print("\nRESUME\n")
    print(pair["resume_text"])

    print("\nJOB DESCRIPTION\n")
    print(pair["job_text"])
```

Historical saved output (not rerun):

```text
PAIR ID: cand_00751:3898162140
JOB TITLE: Business Development Representative

RESUME

Target position: Business Development Executive Objective: Skills: python, machine learning, deep learning, computer vision, numpy, panda, matplotlib, mysql, opencv, pytorch, fast typing skill, ieltsinternet browsing online work ability., nlp, data scrapping, beautiful soup, image processing, satellite imagery Education: ['B.Tech'] ['Electrical'] ['Punjab University'] Bachelor/Honors Experience: 1 to 3 years ['Intern Trainee', 'Intern Developer'] Apparel Sourcing Quality Garment Sourcing Reliable Partner Buyer/Vendor Communication Apparel Sourcing Quality Garment Sourcing Reliable Partner Buyer/Vendor Communication ['Apr 2020', 'Dec 2018'] ['Ongoing', 'Jul 2019']

JOB DESCRIPTION

Job title: Business Development Representative Category: BUSINESS-DEVELOPMENT Skills: b2b sales, cold calling, lead generation, prospecting, salesforce.com, microsoft suite, excel, project management, sales minded, crm tools, competitive personality, forecasting Description: MUST-HAVES * BACHELOR'S DEGREE * EXCELLENT WRITTEN AND VERBAL COMMUNICATION SKILLS * 3+ YEARS B2B SALES, COLD CALLING, LEAD GENERATION, AND PROSPECTING * ABLE TO MANAGE MULTIPLE PROJECTS OR ACCOUNTS SIMULTANEOUSLY * SALES MINDED, DRIVEN, AND COMPETITIVE PERSONALITY PLUSSES * EXPERIENCE LEVERAGING CRM TOOLS, SALESFORCE.COM PREFERRED * PROFICIENT LEVEL AT MICROSOFT SUITE (WORD, POWERPOINT, OUTLOOK), HIGH PROFICIENCY IN EXCEL DAY-TO-DAY INSIGHT GLOBAL'S CLIENT IS SEEKING BUSINESS DEVELOPMENT REPRESENTATIVES TO WORK 5 DAYS ONSITE IN THE BUCKHEAD AREA FOR A DIRECT HIRE OPPORTUNITY. OUR CLIENT IS A LEAD GENERATION, DATA SOLUTIONS, AND DIRECT MARKETING FIRM FOR THE FINANCIAL SERVICES INDUSTRY. AS A BUSINESS DEVELOPMENT REPRESENTATIVE, YOU WILL FOCUS ON BRINGING IN NEW BUSINESS THROUGH COLD CALLING, LEAD GENERATION, AND PROSPECTING; AND WORK CLOSELY WITH ACCOUNT MANAGERS TO CLOSE NEW CLIENTS. THE EXPECTATION IS THAT THIS ROLE GRADUATES TO A FULL ACCOUNT MANAGER WITHIN THE FIRST YEAR AFTER STARTING BASED ON PERFORMANCE. THIS POSITION IS IDEAL FOR SOMEONE WHO THRIVES IN A PROFESSIONAL YET FUN, COMPETITIVE, AND HIGHLY COLLABORATIVE TEAM ENVIRONMENT. PRINCIPAL DUTIES AND RESPONSIBILITIES * INITIATE CUSTOMER CONTACT USING VARIOUS METHODS OF COMMUNICATION INCLUDING LINKEDIN, MINTEL, AND OUR SALESFORCE * COMPLETE INTRODUCTIONS (COLD CALLS), FOLLOW-UPS, DEMONSTRATION CALLS, INTRODUCTION EMAILS, FOLLOW-UP EMAILS, AND DATA AND MARKETING STRATEGY SESSIONS. * WORK WITH ACCOUNT MANAGERS TO DISCUSS AND CLOSE THE BUSINESS YOU BRING IN. * PLAN AND ORGANIZE PERSONAL SALES STRATEGIES INCLUDING DAILY/WEEKLY/MONTHLY SCHEDULE. * MAINTAIN PRODUCT AND INDUSTRY EXPERTISE, INCLUDING NEW PRODUCT OFFERINGS, MARKET CONDITIONS, AND INDUSTRY TRENDS. * PROACTIVELY MANAGE PIPELINE AND OPPORTUNITIES IN SALESFORCE.COM TO MEET AND/OR EXCEED QUOTA AND ACCOUNT PROGRESS UTILIZING SALESFORCE.COM. * EFFECTIVELY FORECAST BUSINESS ON A MONTHLY AND QUARTERLY BASIS.
```
